# Manuscript Memory Engine — Colab model server

Runs **one** vanilla open-source LLM on Colab's GPU via Ollama and exposes it over
a public cloudflared tunnel. Your **local** codebase points at the printed URL and
drives the eval; only LLM calls cross the network.

**Per session:**
1. Pick a GPU runtime (Runtime → Change runtime type → GPU; **A100** for the 72B).
2. **Cell 1** — install Ollama + cloudflared (once per session).
3. **Cell 2** — set `MODEL`, then pull + warm it. Re-run this cell to switch
   models; it frees the previously-loaded one first so you don't run out of RAM.
4. **Cell 3** — open the tunnel and copy the printed **`TUNNEL URL`**. Run it
   **once**; it stays up in the background and the URL does *not* change when you
   switch models with cell 2.
5. On your laptop: set `OLLAMA_BASE_URL=<url>` and `LLM_MODE=ollama` in `.env`,
   then run e.g. `python evals/run_context_experiment.py --model <tag>`.

Small models (≤9B) fit a T4/L4; the 72B needs an A100 (pull it alone, ~47 GB).

In [ ]:
# 1. Install Ollama + download cloudflared (run once per session)
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
print("installed: ollama + cloudflared")

In [ ]:
# 2. Pick a model, (re)start Ollama, free any other loaded model, then pull + warm.
#    Re-run this cell to switch models — the tunnel URL from cell 3 stays the same.
import subprocess, time, requests

MODEL = "qwen2.5:7b"   # <-- change per run: qwen2.5:7b, qwen2.5:72b, llama3.1:8b,
                       #     phi3.5, mistral:7b, yi:9b, gemma2:9b

subprocess.Popen(["ollama", "serve"])   # harmless "address in use" if already running
time.sleep(5)

# free VRAM/RAM from any previously-loaded model (avoids the two-models-at-once crash)
for m in requests.get("http://localhost:11434/api/ps", timeout=10).json().get("models", []):
    if m["name"] != MODEL:
        requests.post("http://localhost:11434/api/generate",
                      json={"model": m["name"], "keep_alive": 0}, timeout=30)

subprocess.run(["ollama", "pull", MODEL])
requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False}, timeout=1200)

loaded = [m["name"] for m in requests.get("http://localhost:11434/api/ps", timeout=10).json().get("models", [])]
print("ready + warmed:", MODEL, "| loaded now:", loaded)

In [ ]:
# 3. Open the tunnel in the background and print the public URL. Run ONCE per
#    session; it keeps running in the background. Copy the TUNNEL URL into your
#    local .env as OLLAMA_BASE_URL. (Re-run only if it prints "not ready yet".)
import subprocess, re, time, pathlib

log = "/content/cf.log"; open(log, "w").close()
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434",
                  "--http-host-header", "localhost:11434"],
                 stdout=open(log, "a"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-\w]+\.trycloudflare\.com", pathlib.Path(log).read_text())
    if m:
        url = m.group(0); break
print("TUNNEL URL:", url or "(not ready yet — just re-run this cell)")